# Lab 1: RAG AI Chatbot with ChromaDB and Qwen

In this lab, you will build a Retrieval-Augmented Generation (RAG) system that:
1. Stores documents in ChromaDB as vector embeddings
2. Retrieves relevant chunks when you ask a question
3. Sends the retrieved context + your question to Qwen2.5-72B-Instruct
4. Returns an answer with source attribution

**Architecture:**
```
User Question → [ChromaDB: retrieve relevant chunks] → [Build prompt with context] → [Qwen2.5-72B] → Answer
```

**What you will learn:**
- How to use ChromaDB as a vector database
- How to split documents into overlapping chunks
- How to build RAG prompts with retrieved context
- How to call LLMs via HuggingFace InferenceClient
- How to build a Gradio chat interface

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
!pip install -q chromadb huggingface_hub gradio python-dotenv

## Step 1: Configuration and Setup

First, we set up the HuggingFace InferenceClient and ChromaDB. The InferenceClient calls the Qwen2.5-72B model hosted on HuggingFace, while ChromaDB stores document embeddings locally.

**Important:** You need a HuggingFace token. Get one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In Google Colab, use `google.colab.userdata` to store your token securely.

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
from huggingface_hub import InferenceClient

# --- Set your HuggingFace token ---
# Option 1: Google Colab (recommended)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    # Option 2: Environment variable or .env file
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv("HF_TOKEN", "")

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"

# Initialize HuggingFace Inference Client
client = InferenceClient(model=MODEL_NAME, token=HF_TOKEN)

# Initialize ChromaDB with default embedding function
chroma_client = chromadb.Client()
embedding_fn = embedding_functions.DefaultEmbeddingFunction()
collection = chroma_client.get_or_create_collection(
    name="rag_documents",
    embedding_function=embedding_fn,
)

print(f"Model: {MODEL_NAME}")
print(f"ChromaDB collection: {collection.name}")
print(f"HF Token: {'configured' if HF_TOKEN else 'NOT SET -- add HF_TOKEN to Colab secrets or .env'}")

## Step 2: Document Chunking

RAG systems split documents into smaller chunks so that only the most relevant pieces are retrieved. Key parameters:
- **Chunk size**: How many characters per chunk (too small = loses context, too large = dilutes relevance)
- **Overlap**: Characters shared between adjacent chunks (prevents splitting important info across chunk boundaries)

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start = end - overlap
    return chunks

# Test chunking with a sample document
sample_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to
natural intelligence displayed by animals and humans. AI research has been defined as
the field of study of intelligent agents, which refers to any system that perceives its
environment and takes actions that maximize its chance of achieving its goals.

The term "artificial intelligence" had previously been used to describe machines that
mimic and display "human" cognitive skills that are associated with the human mind,
such as "learning" and "problem-solving". This definition has since been rejected by
major AI researchers who now describe AI in terms of rationality and acting rationally,
which does not limit how intelligence can be articulated.

Machine learning is a subset of AI that provides systems the ability to automatically
learn and improve from experience without being explicitly programmed. Deep learning
is a subset of machine learning that uses neural networks with many layers.
"""

chunks = chunk_text(sample_text, chunk_size=300, overlap=50)
print(f"Original text length: {len(sample_text)} characters")
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:100] + "...")

## Step 3: Add Documents to ChromaDB

Now we add our chunked documents to ChromaDB. ChromaDB automatically converts text to vector embeddings using its default embedding function and stores them for similarity search.

In [ ]:
# Add the sample document chunks to ChromaDB
chunks = chunk_text(sample_text, chunk_size=500, overlap=50)
ids = [f"sample_chunk_{i}" for i in range(len(chunks))]
metadatas = [{"source": "ai_overview.txt", "chunk_index": i} for i in range(len(chunks))]

collection.upsert(
    documents=chunks,
    ids=ids,
    metadatas=metadatas,
)

# Let's also add a second document about Python
python_text = """
Python is a high-level, general-purpose programming language. Its design philosophy
emphasizes code readability with the use of significant indentation. Python is
dynamically typed and garbage-collected. It supports multiple programming paradigms,
including structured, object-oriented and functional programming.

Python was conceived in the late 1980s by Guido van Rossum at Centrum Wiskunde &
Informatica (CWI) in the Netherlands. Python consistently ranks as one of the most
popular programming languages. It is widely used in data science, machine learning,
web development, automation, and scientific computing.

Key Python libraries for AI include NumPy for numerical computing, Pandas for data
manipulation, Scikit-learn for traditional ML, and PyTorch/TensorFlow for deep learning.
"""

python_chunks = chunk_text(python_text, chunk_size=500, overlap=50)
python_ids = [f"python_chunk_{i}" for i in range(len(python_chunks))]
python_metadatas = [{"source": "python_overview.txt", "chunk_index": i} for i in range(len(python_chunks))]

collection.upsert(
    documents=python_chunks,
    ids=python_ids,
    metadatas=python_metadatas,
)

print(f"Total documents in knowledge base: {collection.count()}")

## Step 4: Query ChromaDB (Retrieval)

Let's test the retrieval part of RAG. We query ChromaDB with a question and it returns the most similar chunks based on vector similarity.

In [ ]:
# Query ChromaDB for relevant chunks
question = "What is machine learning?"
results = collection.query(
    query_texts=[question],
    n_results=3,
)

print(f"Question: {question}\n")
print(f"Retrieved {len(results['documents'][0])} chunks:\n")

for i, (doc, meta, distance) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
):
    print(f"--- Result {i+1} (source: {meta['source']}, distance: {distance:.4f}) ---")
    print(doc[:200] + "..." if len(doc) > 200 else doc)
    print()

## Step 5: RAG Query Pipeline

Now we combine retrieval + generation. The function:
1. Retrieves relevant chunks from ChromaDB
2. Builds a prompt with system message + context + question
3. Calls Qwen2.5-72B via HuggingFace InferenceClient
4. Returns the answer with source attribution

In [ ]:
def rag_query(question, num_results=3, temperature=0.7, max_tokens=500):
    """Query the RAG system: retrieve context, then generate an answer."""
    doc_count = collection.count()

    # Step 1: Retrieve relevant chunks
    context_text = ""
    sources = []
    if doc_count > 0:
        results = collection.query(
            query_texts=[question],
            n_results=min(num_results, doc_count),
        )
        if results["documents"] and results["documents"][0]:
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                context_text += f"\n[Source: {meta['source']}]\n{doc}\n"
                if meta["source"] not in sources:
                    sources.append(meta["source"])

    # Step 2: Build the prompt
    if context_text:
        system_prompt = (
            "You are a helpful AI assistant. Answer the user's question based on "
            "the provided context. If the context doesn't contain relevant information, "
            "say so and answer based on your general knowledge. Always cite which "
            "source document the information comes from."
        )
        user_message = f"Context from knowledge base:\n{context_text}\n\nQuestion: {question}"
    else:
        system_prompt = (
            "You are a helpful AI assistant. The knowledge base is empty. "
            "Answer based on your general knowledge."
        )
        user_message = question

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # Step 3: Call the LLM
    try:
        response = client.chat_completion(
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"Error calling model: {e}"

    # Step 4: Add source attribution
    if sources:
        answer += f"\n\nSources: {', '.join(sources)}"

    return answer

# Test the RAG pipeline
question = "What is the relationship between AI and machine learning?"
print(f"Question: {question}\n")
answer = rag_query(question)
print(f"Answer:\n{answer}")

## Step 6: Try More Questions

Test the RAG system with different questions. Notice how it retrieves different chunks depending on the question, and cites the source documents.

In [ ]:
# Try questions that target different documents
questions = [
    "Who created Python?",
    "What is deep learning?",
    "What Python libraries are used for AI?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_query(q)}")
    print("-" * 60)

## Step 7: Launch Gradio Chat Interface

Finally, we wrap everything in a Gradio Blocks interface with:
- A **Chat tab** with multi-turn conversation and configurable settings
- A **Knowledge Base tab** for uploading documents

Run the cell below to launch the interactive chatbot.

In [ ]:
import gradio as gr


def upload_documents(files, chunk_size, chunk_overlap):
    """Process uploaded files and add to ChromaDB."""
    if not files:
        return "No files uploaded."

    chunk_size = int(chunk_size)
    chunk_overlap = int(chunk_overlap)
    total_chunks = 0

    for file in files:
        try:
            with open(file.name, "r", encoding="utf-8") as f:
                text = f.read()
        except UnicodeDecodeError:
            with open(file.name, "r", encoding="latin-1") as f:
                text = f.read()

        filename = os.path.basename(file.name)
        chunks = chunk_text(text, chunk_size=chunk_size, overlap=chunk_overlap)

        if chunks:
            ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
            metadatas = [{"source": filename, "chunk_index": i} for i in range(len(chunks))]
            collection.upsert(documents=chunks, ids=ids, metadatas=metadatas)
            total_chunks += len(chunks)

    doc_count = collection.count()
    return f"Processed {len(files)} file(s), added {total_chunks} chunks.\nTotal documents in knowledge base: {doc_count}"


def clear_knowledge_base():
    """Clear all documents from ChromaDB."""
    global collection
    chroma_client.delete_collection("rag_documents")
    collection = chroma_client.get_or_create_collection(
        name="rag_documents",
        embedding_function=embedding_fn,
    )
    return "Knowledge base cleared."


def query_rag_chat(question, num_results, temperature, max_tokens, history):
    """Chat function for Gradio chatbot."""
    history = history or []
    if not question or not question.strip():
        return history, ""

    answer = rag_query(question, int(num_results), temperature, int(max_tokens))
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    return history, ""


# Build the Gradio interface
with gr.Blocks(title="RAG AI Chatbot") as demo:
    gr.Markdown(
        "# RAG AI Chatbot -- Qwen + ChromaDB\n"
        "Upload documents to build a knowledge base, then chat with your data "
        "using Retrieval-Augmented Generation powered by Qwen2.5-72B and ChromaDB."
    )

    with gr.Tab("Chat"):
        chatbot = gr.Chatbot(label="Conversation", height=400)
        with gr.Row():
            question_input = gr.Textbox(
                label="Ask a question",
                placeholder="Type your question here...",
                scale=4,
            )
            send_btn = gr.Button("Send", variant="primary", scale=1)

        with gr.Accordion("Settings", open=False):
            with gr.Row():
                num_results = gr.Slider(1, 10, value=3, step=1, label="Context chunks to retrieve")
                temperature = gr.Slider(0.0, 1.0, value=0.7, step=0.1, label="Temperature")
                max_tokens = gr.Slider(100, 2000, value=500, step=100, label="Max Tokens")

        clear_chat_btn = gr.Button("Clear Chat")

        send_btn.click(
            fn=query_rag_chat,
            inputs=[question_input, num_results, temperature, max_tokens, chatbot],
            outputs=[chatbot, question_input],
        )
        question_input.submit(
            fn=query_rag_chat,
            inputs=[question_input, num_results, temperature, max_tokens, chatbot],
            outputs=[chatbot, question_input],
        )
        clear_chat_btn.click(fn=lambda: ([], ""), outputs=[chatbot, question_input])

    with gr.Tab("Knowledge Base"):
        gr.Markdown("### Upload Documents\nUpload `.txt`, `.md`, or `.csv` files to build the knowledge base.")
        with gr.Row():
            file_upload = gr.File(label="Upload files", file_count="multiple", file_types=[".txt", ".md", ".csv"])
            with gr.Column():
                chunk_size = gr.Slider(100, 2000, value=500, step=100, label="Chunk Size (characters)")
                chunk_overlap = gr.Slider(0, 200, value=50, step=10, label="Chunk Overlap (characters)")

        upload_btn = gr.Button("Upload & Process", variant="primary")
        upload_status = gr.Textbox(label="Status", interactive=False)
        upload_btn.click(fn=upload_documents, inputs=[file_upload, chunk_size, chunk_overlap], outputs=[upload_status])

        gr.Markdown("---")
        clear_kb_btn = gr.Button("Clear Knowledge Base", variant="stop")
        clear_kb_status = gr.Textbox(label="Status", interactive=False)
        clear_kb_btn.click(fn=clear_knowledge_base, outputs=[clear_kb_status])

demo.launch()

## Key Takeaways

1. **RAG = Retrieval + Generation** -- ChromaDB finds relevant documents, the LLM generates answers using that context
2. **Chunking matters** -- chunk size and overlap affect retrieval quality; too small loses context, too large dilutes relevance
3. **Embeddings are the backbone** -- ChromaDB's default embedding function converts text to vectors for similarity search
4. **Source attribution builds trust** -- showing which documents were used helps users verify answers
5. **Gradio makes it interactive** -- from notebook prototype to web app in a single cell

## Experiments to Try

- Upload your own `.txt` or `.md` files and ask questions about them
- Try different chunk sizes (200 vs 1000) and see how answers change
- Set temperature to 0.0 for deterministic answers vs 1.0 for creative ones
- Ask a question that the knowledge base doesn't cover and see how the model responds